# Retrain the 6,023-example Primary-Maths Manim dataset

This notebook reproduces supervised fine-tuning from the original Qwen2.5-Coder 3B base with either equivalent 6,023-row dataset file: `primary_maths_manim_qwen_messages_final_6023.jsonl` or `primary_maths_manim_sharegpt_final_6023.jsonl`. It saves checkpoints to Google Drive and exports a small report bundle containing the exact configuration, dataset hash, training history, validation results, runtime, and a loss chart.

Before starting, select **Runtime → Change runtime type → GPU**. Run the cells in order. Do not use the current calculus checkpoint as the starting model if the goal is to reproduce the original 6,023-example SFT experiment.

In [ ]:
# Install a mutually compatible Unsloth training stack. The exact versions
# selected in this runtime are recorded in sft_report_summary.json.
%pip install -q --upgrade unsloth

In [ ]:
import torch

assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name} ({vram_gb:.1f} GiB VRAM)")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Upload the two required project files

Upload these files from this workspace:

1. `train_unsloth_sft.py`
2. Either `primary_maths_manim_qwen_messages_final_6023.jsonl` **or** `primary_maths_manim_sharegpt_final_6023.jsonl` (about 14 MB). Both contain the same 6,023 conversations in formats supported by the trainer.

In [ ]:
from google.colab import files
from pathlib import Path

trainer_path = Path("/content/train_unsloth_sft.py")
dataset_candidates = [
    Path("/content/primary_maths_manim_qwen_messages_final_6023.jsonl"),
    Path("/content/primary_maths_manim_sharegpt_final_6023.jsonl"),
]

# Reuse files already saved by a previous upload attempt.
if not trainer_path.is_file() or not any(path.is_file() for path in dataset_candidates):
    uploaded = files.upload()
    for filename, payload in uploaded.items():
        Path("/content", filename).write_bytes(payload)

if not trainer_path.is_file():
    raise ValueError("Missing train_unsloth_sft.py")
DATASET_PATH = next((path for path in dataset_candidates if path.is_file()), None)
if DATASET_PATH is None:
    raise ValueError("Upload either the Qwen messages or ShareGPT 6,023-row JSONL file.")
print(f"Trainer: {trainer_path.name}")
print(f"Dataset: {DATASET_PATH.name}")

## Preflight the exact dataset

This fails before GPU allocation if the upload is incomplete, has the wrong row count, contains malformed JSON, or does not follow the required `system → user → assistant` structure. It recognizes the recorded SHA-256 fingerprint of either equivalent dataset format.

In [ ]:
import hashlib
import json

EXPECTED_ROWS = 6023
EXPECTED_SHA256 = {
    "primary_maths_manim_qwen_messages_final_6023.jsonl": "1a784d85752316042438700de8c83ab5d27d246cb990b09789ac88dd34cc8fbf",
    "primary_maths_manim_sharegpt_final_6023.jsonl": "9af06d285a8f282db0572f0cd75a5fa1d5bb8711e125c02ca72e01ec8b68b012",
}

digest = hashlib.sha256()
with DATASET_PATH.open("rb") as source:
    for chunk in iter(lambda: source.read(1024 * 1024), b""):
        digest.update(chunk)
actual_sha256 = digest.hexdigest()

row_count = 0
with DATASET_PATH.open(encoding="utf-8") as source:
    for line_number, line in enumerate(source, start=1):
        if not line.strip():
            continue
        row = json.loads(line)
        messages = row.get("messages")
        if messages is not None:
            roles = [message.get("role") for message in messages]
            contents = [message.get("content") for message in messages]
        else:
            conversations = row.get("conversations") or []
            role_map = {"system": "system", "human": "user", "gpt": "assistant"}
            roles = [role_map.get(message.get("from")) for message in conversations]
            contents = [message.get("value") for message in conversations]
        if roles != ["system", "user", "assistant"]:
            raise ValueError(f"Line {line_number} has invalid roles: {roles}")
        if any(not isinstance(content, str) or not content.strip() for content in contents):
            raise ValueError(f"Line {line_number} contains empty message content")
        row_count += 1

assert row_count == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS} rows, found {row_count}"
expected_sha256 = EXPECTED_SHA256.get(DATASET_PATH.name)
assert expected_sha256 and actual_sha256 == expected_sha256, (
    f"Dataset hash mismatch. Expected {expected_sha256}, found {actual_sha256}"
)
print(f"Validated {row_count:,} rows")
print(f"SHA-256: {actual_sha256}")

## Experiment configuration

The default starts from the original code-specialized 3B base, trains a rank-32 rsLoRA adapter for three epochs, uses a deterministic 95/5 split, and reloads the checkpoint with the lowest validation loss. The conservative `2 × 8` batch setup has the same effective batch size of 16 as `4 × 4`, but is safer on a 16 GB T4.

Give every independent experiment a new `RUN_NAME`. Set `RESUME = True` only when continuing checkpoints already present in that same Drive folder.

In [ ]:
MODEL_ID = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"
RUN_NAME = "primary-maths-6023-run-01"
OUTPUT_DIR = f"/content/drive/MyDrive/manim-sft/{RUN_NAME}"

MAX_SEQ_LENGTH = 1280
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
EPOCHS = 3
LEARNING_RATE = 1e-4
VALIDATION_SIZE = 0.05
LOGGING_STEPS = 10
SEED = 3407
RESUME = False

# Optional Hugging Face upload. Leave as None for a Drive-only run.
HUB_MODEL_ID = None  # Example: "YOUR_USERNAME/qwen-primary-maths-manim-sft"
print(f"Output directory: {OUTPUT_DIR}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

## Train

Training runs in the notebook kernel and displays a live table. Checkpoints, `trainer_state.json`, raw metrics, `training_history.csv`, `sft_report_summary.json`, and the final LoRA adapter are written directly to Drive. On a Colab T4 this is a multi-hour run; Drive checkpoints allow recovery after a disconnect.

In [ ]:
import runpy
import sys

if HUB_MODEL_ID:
    from huggingface_hub import login
    login()

trainer_argv = [
    "train_unsloth_sft.py",
    "--model-id", MODEL_ID,
    "--local-dataset-file", str(DATASET_PATH),
    "--output-dir", OUTPUT_DIR,
    "--max-seq-length", str(MAX_SEQ_LENGTH),
    "--batch-size", str(BATCH_SIZE),
    "--eval-batch-size", str(BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION_STEPS),
    "--epochs", str(EPOCHS),
    "--learning-rate", str(LEARNING_RATE),
    "--validation-size", str(VALIDATION_SIZE),
    "--logging-steps", str(LOGGING_STEPS),
    "--checkpoint-strategy", "epoch",
    "--packing",
    "--live-table",
    "--seed", str(SEED),
]
if RESUME:
    trainer_argv.append("--resume-from-checkpoint")
if HUB_MODEL_ID:
    trainer_argv.extend(["--hub-model-id", HUB_MODEL_ID])

print("Starting the 6,023-example SFT run.")
original_argv = sys.argv.copy()
try:
    sys.argv = trainer_argv
    runpy.run_path("/content/train_unsloth_sft.py", run_name="__main__")
finally:
    sys.argv = original_argv

## Build and download the report-data bundle

This cell is safe to rerun after reconnecting to Colab. It reads the saved Drive artifacts, creates a loss chart and a one-row results table, and downloads a compact ZIP. Model weights and checkpoints are intentionally excluded from the ZIP.

In [ ]:
import json
import math
import shutil

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

output_path = Path(OUTPUT_DIR)
summary_path = output_path / "sft_report_summary.json"
history_path = output_path / "training_history.csv"
if not summary_path.is_file() or not history_path.is_file():
    raise FileNotFoundError("Training report files are missing; finish training first.")

summary = json.loads(summary_path.read_text(encoding="utf-8"))
history = pd.read_csv(history_path)
train_history = history.dropna(subset=["loss"]) if "loss" in history else pd.DataFrame()
eval_history = history.dropna(subset=["eval_loss"]) if "eval_loss" in history else pd.DataFrame()

fig, axis = plt.subplots(figsize=(9, 5))
if not train_history.empty:
    axis.plot(train_history["step"], train_history["loss"], label="training loss", alpha=0.85)
if not eval_history.empty:
    axis.plot(eval_history["step"], eval_history["eval_loss"], "o-", label="validation loss", linewidth=2)
axis.set(title="Primary-maths 6,023-example SFT loss", xlabel="Optimizer step", ylabel="Completion-token NLL")
axis.grid(alpha=0.25)
axis.legend()
fig.tight_layout()
loss_chart = output_path / "sft_loss_curve.png"
fig.savefig(loss_chart, dpi=180)
plt.show()

eval_metrics = summary["result"].get("eval_metrics") or {}
train_metrics = summary["result"].get("train_metrics") or {}
eval_loss = eval_metrics.get("eval_loss")
report_row = {
    "run_name": RUN_NAME,
    "base_model": summary["model"]["model_id"],
    "dataset_rows_total": summary["dataset"]["train_rows"] + summary["dataset"]["validation_rows"],
    "train_rows": summary["dataset"]["train_rows"],
    "validation_rows": summary["dataset"]["validation_rows"],
    "dataset_sha256": summary["dataset"].get("sha256"),
    "global_step": summary["result"]["global_step"],
    "best_checkpoint": summary["result"].get("best_model_checkpoint"),
    "best_validation_loss": summary["result"].get("best_metric"),
    "final_selected_validation_loss": eval_loss,
    "validation_perplexity": math.exp(eval_loss) if isinstance(eval_loss, (int, float)) and eval_loss < 50 else None,
    "train_runtime_seconds": train_metrics.get("train_runtime"),
    "train_samples_per_second": train_metrics.get("train_samples_per_second"),
    "effective_batch_size": summary["derived_configuration"]["effective_batch_size_one_gpu"],
}
report_table = pd.DataFrame([report_row])
report_table_path = output_path / "sft_report_table.csv"
report_table.to_csv(report_table_path, index=False)
display(report_table.T.rename(columns={0: "value"}))

bundle_dir = output_path / "report_data"
bundle_dir.mkdir(parents=True, exist_ok=True)
artifact_names = [
    "sft_report_summary.json",
    "training_history.csv",
    "sft_report_table.csv",
    "sft_loss_curve.png",
    "trainer_state.json",
    "train_results.json",
    "eval_results.json",
]
for name in artifact_names:
    source = output_path / name
    if source.is_file():
        shutil.copy2(source, bundle_dir / name)
archive_base = output_path / f"{RUN_NAME}-report-data"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=bundle_dir))
print(f"Report bundle: {archive_path}")
files.download(str(archive_path))

## What these results support

The ZIP supports reporting the exact dataset, configuration, training runtime, convergence, validation loss, and selected checkpoint. It does **not** by itself establish Manim render accuracy or mathematical improvement. For a quality claim, generate from the untouched base model and this run's `final_adapter` on the same held-out prompts in `primary_maths_manim_evals_150.jsonl`, then render and score both sets with the same verifier. Keep those held-out prompts out of training.

## Export Q4_K_M GGUF and push to Hugging Face

Upload `export_unsloth_gguf.py` from the project. The exporter loads `OUTPUT_DIR/final_adapter`, merges it into the recorded base model, creates a standalone Q4_K_M GGUF in temporary Colab storage, calculates SHA-256, writes an export manifest and model card, uploads the artifacts, and verifies that the GGUF is visible in the destination repository.

A private repository is the default. Set `GGUF_PRIVATE = False` only if you intend to publish the model. Use a Hugging Face token with permission to create and write model repositories. The export needs substantial temporary disk space and can take tens of minutes.

In [ ]:
from google.colab import files
from pathlib import Path

exporter_path = Path("/content/export_unsloth_gguf.py")
if not exporter_path.is_file():
    uploaded = files.upload()
    if exporter_path.name not in uploaded:
        raise ValueError(f"Upload {exporter_path.name}")
    exporter_path.write_bytes(uploaded[exporter_path.name])
print(f"Exporter ready: {exporter_path}")

In [ ]:
import json
from huggingface_hub import login, whoami

login()  # Paste a Hugging Face write token when prompted.
HF_OWNER = whoami()["name"]
GGUF_REPO_NAME = "qwen-primary-maths-manim-6023-Q4_K_M-GGUF"
GGUF_REPO_ID = f"{HF_OWNER}/{GGUF_REPO_NAME}"
GGUF_PRIVATE = True

ADAPTER_PATH = Path(OUTPUT_DIR) / "final_adapter"
GGUF_OUTPUT_DIR = Path("/content") / f"{RUN_NAME}-Q4_K_M-GGUF"
summary = json.loads((Path(OUTPUT_DIR) / "sft_report_summary.json").read_text(encoding="utf-8"))
BASE_MODEL_ID = summary["model"]["model_id"]
DATASET_LABEL = summary["dataset"].get("filename") or "primary-maths-manim-6023"
DATASET_SHA256 = summary["dataset"].get("sha256") or "unknown"

assert (ADAPTER_PATH / "adapter_config.json").is_file(), f"Missing adapter: {ADAPTER_PATH}"
assert list(ADAPTER_PATH.glob("adapter_model*")), f"Missing adapter weights: {ADAPTER_PATH}"
print(f"Source adapter: {ADAPTER_PATH}")
print(f"Destination: https://huggingface.co/{GGUF_REPO_ID}")
print(f"Repository privacy: {'private' if GGUF_PRIVATE else 'public'}")

In [ ]:
import gc
import runpy
import sys
import torch

gc.collect()
torch.cuda.empty_cache()

export_argv = [
    "export_unsloth_gguf.py",
    "--model-path", str(ADAPTER_PATH),
    "--hub-model-id", GGUF_REPO_ID,
    "--output-dir", str(GGUF_OUTPUT_DIR),
    "--quantization", "q4_k_m",
    "--max-seq-length", str(MAX_SEQ_LENGTH),
    "--maximum-memory-usage", "0.5",
    "--base-model-id", BASE_MODEL_ID,
    "--dataset-id", "",
    "--dataset-label", DATASET_LABEL,
    "--dataset-sha256", DATASET_SHA256,
]
if GGUF_PRIVATE:
    export_argv.append("--private")

original_argv = sys.argv.copy()
try:
    sys.argv = export_argv
    runpy.run_path(str(exporter_path), run_name="__main__")
finally:
    sys.argv = original_argv